# Lab 11 · Kỷ luật đo lường cho LLM: schema, nhãn tay & hậu kiểm

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 11**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng bài 11 chạy toàn bộ pipeline Gemini trên 24 đánh giá. Lab này tập trung vào
ba kỹ năng cốt lõi của hợp phần LLM và không yêu cầu khóa API: bạn sẽ chọn mẫu phù hợp,
viết schema, rồi **đánh giá một bộ đầu ra LLM mô phỏng có lỗi cài sẵn** bằng nhãn tay.
Đây cũng là quy trình mỗi nhóm sẽ thực hiện với ít nhất 100 nhãn trong bài tập lớn.

*Lab khoảng 50 phút; 30 phút cuối dành cho phần hỗ trợ bài tập lớn ở cuối notebook.*

## Cách làm việc trong bài lab

- Bài tập được chia thành các bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`. Khi tất cả
  `assert` chạy thành công, lời giải đã đáp ứng yêu cầu của bước đó.
- Với phần khởi động và bài có hướng dẫn, bạn nên **tự gõ, không dùng AI**. Các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết sẽ kiểm tra những kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa tiến triển sau 3 phút ở một bước, hãy trao đổi với giảng viên thực hành.

## Mục tiêu

Sau bài lab, bạn:

1. Chọn mẫu đánh giá phù hợp để gửi LLM và ước lượng chi phí token trước khi gọi.
2. Viết schema Pydantic có enum và kiểm tra cách schema chặn nhãn bịa **trước khi** vào bảng.
3. Đo độ chính xác của đầu ra LLM trên nhãn tay và phân loại lỗi ngữ nghĩa.
4. Viết một quy tắc hậu kiểm tự động.

In [ ]:
%pip install -q pydantic

## Phần 0 · Chọn mẫu & ước chi phí (~12 phút)

Gói miễn phí có giới hạn, vì vậy không nên gửi toàn bộ 690 nghìn đánh giá cho mô hình. Chọn mẫu
là một bước cần thiết của pipeline; lab 7 cho thấy khoảng 59 nghìn bình luận quá ngắn để phân tích.

In [ ]:
import pandas as pd

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/reviews.csv.gz")
# Lưu ý: ô này tải ~59MB (toàn bộ reviews.csv.gz) rồi mới lấy mẫu — chờ 10–30s, KHÔNG phải treo.
# (Muốn nhanh hơn: lưu file về Drive một lần rồi đọc lại từ Drive.)
rv = pd.read_csv(URL, usecols=["id", "comments"]).dropna(subset=["comments"])
rv["comments"] = rv["comments"].str.replace("<br/>", " ", regex=False)

# TODO: lọc bình luận dài >= 50 ký tự, rồi lấy mẫu 100 đánh giá với random_state=42
du_dai = ...
mau = ...

# --- Ô kiểm tra ---
assert len(mau) == 100
assert mau["comments"].str.len().min() >= 50
print(f"Chọn 100/{len(du_dai):,} đánh giá đủ dài — mẫu tái lập được nhờ random_state.")

In [ ]:
# Ước lượng token: quy tắc ngón tay cái ~4 ký tự/token (đủ cho ước lượng chi phí)
# TODO: tính tổng token ước lượng của 100 comment trong mẫu (cộng len // 4)
tong_token = ...

# Giá flash-lite: $0.25 / 1M token vào. TODO: ước chi phí phần dữ liệu (USD)
chi_phi = ...

# --- Ô kiểm tra ---
assert tong_token == int((mau["comments"].str.len() // 4).sum())
assert chi_phi < 0.01
print(f"~{tong_token:,} token dữ liệu ≈ ${chi_phi:.5f} — chưa tính prompt; mẫu 100 gần như miễn phí.")

## Phần 1 · Schema là tầng kiểm tra đầu tiên (~12 phút)

In [ ]:
from pydantic import BaseModel, ValidationError
from typing import Literal

Aspect = Literal["location", "cleanliness", "host", "noise", "amenities", "value"]

# TODO: viết class ReviewInfo(BaseModel) gồm 4 trường:
#   sentiment: Literal 3 giá trị "positive" / "mixed" / "negative"
#   aspects_positive: list[Aspect]
#   aspects_negative: list[Aspect]
#   language: str
class ReviewInfo(BaseModel):
    ...

# --- Ô kiểm tra ---
ok = ReviewInfo.model_validate_json(
    '{"sentiment": "mixed", "aspects_positive": ["location"], '
    '"aspects_negative": ["noise"], "language": "en"}')
assert ok.sentiment == "mixed" and ok.aspects_negative == ["noise"]
try:
    ReviewInfo.model_validate_json(
        '{"sentiment": "happy", "aspects_positive": [], "aspects_negative": [], "language": "en"}')
    raise SystemError("Lẽ ra phải bị chặn!")
except ValidationError:
    print("Schema chặn đúng nhãn 'happy' ngoài enum — tầng kiểm tra hoạt động.")

## Phần 2 · Đánh giá đầu ra LLM bằng nhãn tay (~26 phút)

Dưới đây là **10 đánh giá thật** tại Santiago, bộ nhãn tay `GOLD` và **10 đầu ra LLM mô phỏng**.
Một số lỗi đã được cài sẵn để mô phỏng dữ liệu thực tế. Nhiệm vụ của bạn là dùng schema
và nhãn tay để phát hiện các lỗi này.

In [ ]:
REVIEWS = {
 1: "Place is great for a family, all clean, good location, bit noisy as the main avenue is behind, but we had a great time.",
 2: "Apartment was irrelevant with pictures and not clean too so we cancelled our reservation. Alvaro helped us about cancellation process.",
 3: "Cristian fue muy amable en todo momento, el lugar como se describia. La zona con muy buena movilidad. Muy recomendable. Gracias Cristian",
 4: "Hermoso alojamiento! Lo pasamos re bien mi hija y yo. Es un poco ruidosa la zona si se abre la ventana. La vista es bella y esta muy bien ubicado. Sin duda volveriamos",
 5: "Great location but apartment needs some attention to detail. Cable TV and wifi was out of service. Communication with host was poor. Bedding was not optimal.",
 6: "Good WiFi, great location and centrally located. Shower was hot and had good pressure and there was enough space for two people for 6 days.",
 7: "Es tal cual las fotos, buena ubicacion, tranquilo y sin ruido. Es en un piso 15 por si le temen a las alturas.",
 8: ".",
 9: "Nice place in cool region. Very noisy environment and apartment is not very clean.",
 10: "location was great, wifi was spotty. But overall not a bad place.",
}

# Nhãn tay của nhóm (GOLD) — quy ước: đánh giá rỗng/vô nghĩa -> mixed, không khía cạnh, "und"
GOLD = {
 1: {"sentiment": "mixed",    "language": "en"},
 2: {"sentiment": "negative", "language": "en"},
 3: {"sentiment": "positive", "language": "es"},
 4: {"sentiment": "mixed",    "language": "es"},
 5: {"sentiment": "negative", "language": "en"},
 6: {"sentiment": "positive", "language": "en"},
 7: {"sentiment": "positive", "language": "es"},
 8: {"sentiment": "mixed",    "language": "und"},
 9: {"sentiment": "mixed",    "language": "en"},
 10: {"sentiment": "mixed",   "language": "en"},
}

# Đầu ra LLM mô phỏng (JSON thô) — CÓ LỖI CÀI SẴN, không sửa
LLM_OUT = {
 1: '{"sentiment": "mixed", "aspects_positive": ["cleanliness", "location"], "aspects_negative": ["noise"], "language": "en"}',
 2: '{"sentiment": "negative", "aspects_positive": ["host"], "aspects_negative": ["photos", "cleanliness"], "language": "en"}',
 3: '{"sentiment": "positive", "aspects_positive": ["host", "location"], "aspects_negative": [], "language": "es"}',
 4: '{"sentiment": "positive", "aspects_positive": ["location"], "aspects_negative": [], "language": "es"}',
 5: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["amenities", "host"]}',
 6: '{"sentiment": "positive", "aspects_positive": ["amenities", "location"], "aspects_negative": [], "language": "en"}',
 7: '{"sentiment": "positive", "aspects_positive": ["location"], "aspects_negative": [], "language": "es"}',
 8: '{"sentiment": "mixed", "aspects_positive": [], "aspects_negative": [], "language": "und"}',
 9: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["noise", "cleanliness"], "language": "en"}',
 10: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["amenities"], "language": "en"}',
}
print(len(REVIEWS), "đánh giá · 10 đầu ra mô phỏng · 10 nhãn tay")

### Bước 1 · Kiểm tra hàng loạt — schema phát hiện được gì?

In [ ]:
# TODO: duyệt LLM_OUT, validate từng JSON bằng ReviewInfo;
#       hợp lệ -> bỏ vào dict hop_le[id], lỗi -> bỏ id vào list loi_schema
hop_le = {}
loi_schema = []
for rid, raw in LLM_OUT.items():
    ...

# --- Ô kiểm tra ---
assert len(hop_le) == 8 and sorted(loi_schema) == [2, 5]
print(f"Schema chặn {len(loi_schema)} đầu ra: ID {loi_schema} — xem lỗi ở cell sau.")

In [ ]:
# Xem lý do từng đầu ra bị chặn
for rid in loi_schema:
    try:
        ReviewInfo.model_validate_json(LLM_OUT[rid])
    except ValidationError as e:
        print(f"--- id {rid}: {e.errors()[0]['type']} tại {e.errors()[0]['loc']}")

ID 2 tạo nhãn `photos` ngoài enum; ID 5 **thiếu trường** `language`. Cả hai bị chặn
*trước khi* vào bảng, hiệu quả hơn so với làm sạch sau đó. Trong pipeline thực tế, các đầu ra
bị chặn sẽ được **thử lại** hoặc ghi vào danh sách lỗi.

### Bước 2 · Độ chính xác của nhãn cảm xúc

In [ ]:
# TODO: trên 8 đầu ra hợp lệ, đếm số ID có sentiment TRÙNG với GOLD; tính độ chính xác
so_dung = ...
acc = ...

# --- Ô kiểm tra ---
assert so_dung == 5 and acc == 0.625
sai = [rid for rid in hop_le if hop_le[rid].sentiment != GOLD[rid]["sentiment"]]
print(f"Accuracy sentiment: {acc:.0%} — sai ở id {sorted(sai)}.")

### Bước 3 · Nhìn vào chỗ sai — phân loại lỗi

Đọc lại ba đánh giá bị gán sai bằng cách chạy cell dưới, rồi đối chiếu bảng phân loại:

| ID | GOLD | LLM | Kiểu lỗi |
|---|---|---|---|
| 4 | mixed | positive | **bỏ sót ý chê** — lời khen dài che khuất nhận xét về tiếng ồn |
| 9 | mixed | negative | **trường hợp khó phân định** — nhóm gán mixed ("nice place" + 2 ý chê) |
| 10 | mixed | negative | **hiểu sai phủ định** — "not a *bad* place" là khen nhẹ, mô hình lại đọc thành chê |

Trường hợp 9 cho thấy người gán nhãn cũng có thể không thống nhất với nhau. Vì vậy, hướng dẫn
gán nhãn đã được nhóm thống nhất trước là **mốc tham chiếu** và phải được áp dụng nhất quán.

In [ ]:
for rid in [4, 9, 10]:
    print(f"[{rid}] GOLD={GOLD[rid]['sentiment']:8s} LLM={hop_le[rid].sentiment:8s} | {REVIEWS[rid][:90]}")

### Bước 4 · Hậu kiểm tự động

In [ ]:
# Quy tắc hậu kiểm: đầu ra nào tự mâu thuẫn — sentiment "positive" mà lại có
# aspects_negative không rỗng (hoặc "negative" mà có aspects_positive không rỗng)?
# TODO: thu ID của các đầu ra hợp lệ vi phạm quy tắc trên vào list mau_thuan
mau_thuan = ...

# --- Ô kiểm tra ---
assert sorted(mau_thuan) == [9, 10]
print(f"Hậu kiểm bắt id {sorted(mau_thuan)} — trùng với 2/3 ca gán sai ở Bước 3!")

Một quy tắc hậu kiểm ba dòng, không cần nhãn tay, đã gắn cờ hai trong ba trường hợp sai
để người phụ trách đọc lại. Nhãn tay đo *chất lượng tổng thể*; hậu kiểm chạy *trên từng đầu ra mới*
khi pipeline vận hành. Hợp phần LLM của bài tập lớn cần **cả hai**.

## Bài tự làm ✅ mở · Chạy thật với khóa API của bạn

Nếu đã có khóa API, hãy dùng mã nguồn mẫu ở mục 4 của notebook bài giảng bài 11: đầu ra có cấu trúc
với `ReviewInfo.model_json_schema()`. Gọi Gemini cho **10 đánh giá trong `REVIEWS`**, rồi:
(1) kiểm tra schema như Bước 1; (2) đo độ chính xác với `GOLD` như Bước 2; (3) so sánh kết quả
của mô hình thật với bộ mô phỏng trong lab. Dùng `time.sleep(4)` giữa các lời gọi và lưu cache JSON.
Ghi lại các ID được gán đúng hoặc sai; mô hình có hiểu sai câu phủ định ở ID 10 không?

In [ ]:
# Viết bài tự làm của bạn ở đây (cần GEMINI_API_KEY trong Colab Secrets)

---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm theo nhóm)

Trọng tâm tuần này: **thiết lập hợp phần LLM có thể kiểm chứng và chạy lại được.**

1. ☐ Khóa API Gemini của người phụ trách nằm trong **Colab Secrets / biến môi trường** —
   tuyệt đối chưa từng xuất hiện trong code hay commit nào (soát bằng `git log -p | grep`).
2. ☐ **Mẫu đánh giá** để gán nhãn đã chốt: cách lọc theo độ dài và thời gian, cùng `random_state`,
   đủ ≥100, cân nhắc tỷ lệ ngôn ngữ.
3. ☐ **Schema + enum khía cạnh** bản nháp đã viết (theo lab này); danh mục khía cạnh
   hợp với câu hỏi phân tích của nhóm.
4. ☐ **Hướng dẫn gán nhãn** 5–10 dòng đã viết chung (quy ước trường hợp rỗng, mixed và đa
   ngôn ngữ); mỗi đánh giá do hai người gán độc lập, sau đó thảo luận các trường hợp bất đồng.
5. ☐ Phương pháp đối chứng không dùng LLM (từ khoá bài 7) chạy được trên đúng mẫu để so sánh.

> Nhóm hoàn thành sớm: chạy thử 5 lời gọi Gemini với schema của nhóm và kiểm tra xem đầu ra thực tế
> có đúng cấu trúc hay không.

## Tóm tắt bài lab

| Bạn đã làm | Dùng cho |
|---|---|
| Chọn mẫu + ước token trước khi gọi | kế hoạch chi phí cho hợp phần LLM |
| Schema/enum chặn nhãn bịa và trường thiếu | pipeline LLM của nhóm |
| Độ chính xác trên nhãn tay + phân loại lỗi | mục "đo chất lượng trên ≥100 nhãn" của đề |
| Hậu kiểm tự mâu thuẫn | vận hành pipeline sau khi nộp |

Bài giảng tiếp theo sẽ dùng **trực quan hoá cơ bản** để biểu diễn các con số này.